# Parte 3 · Modelado ML

Entrena un modelo que prediga `fla_churn90` con el dataset proporcionado.

**Requisitos**:

1. Split estratificado y **temporal-aware**. Justifica cómo evitas leakage.
2. Pipeline con `ColumnTransformer` y al menos un modelo (LogReg o XGBoost/LightGBM).
3. Métricas correctas para clasificación binaria **desbalanceada**: AUC, PR-AUC, recall@k, calibración.
4. Interpretabilidad: top-5 features importantes con justificación. SHAP valorable.
5. Documenta en `DECISIONS.md`: ¿qué features descartaste y por qué? ¿hay alguna que sospeches que sea trampa?

> Si tu modelo da AUC > 0.95 en test, **investiga antes de celebrarlo**. Es muy probable leakage.

In [1]:
import math
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, precision_recall_curve
)
import lightgbm as lgb
import shap
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

DATA_PATH = Path('../data/transactions_sample.csv')
OUTPUTS = Path('../outputs')
OUTPUTS.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

## 1 · Carga y EDA mínimo

Reutiliza `load_clean` de la Parte 1 (o carga aquí y limpia). Antes de modelar, mira los datos.

In [2]:
import sys
sys.path.insert(0, "../..")
from src.parte1_pandas import load_clean

df = load_clean(DATA_PATH)

print(f"Rows: {len(df):,}  |  Merchants: {df['merchant_id'].nunique():,}")
print(f"Churn rate: {df.drop_duplicates('merchant_id')['fla_churn90'].mean():.2%}")
print(f"Date range: {df['transaction_date'].min().date()} → {df['transaction_date'].max().date()}")
print(f"Reference date: {df['reference_date'].max().date()}")

Rows: 199,818  |  Merchants: 9,982
Churn rate: 8.75%
Date range: 2024-01-01 → 2025-12-31
Reference date: 2025-09-30


## 2 · Feature engineering y selección

Documenta en `DECISIONS.md` qué features incluyes y cuáles descartas, **y por qué**.

Pista: hay al menos una columna en el CSV que **no deberías usar** como feature aunque parezca útil. Piensa si la información está disponible *en el momento de la predicción*.

In [3]:
REF_DATE  = df["reference_date"].max()
WIN_3M    = REF_DATE - pd.Timedelta(days=90)   # 2025-07-01 approx
WIN_6M    = REF_DATE - pd.Timedelta(days=180)  # 2025-04-03 approx

# ── Only pre-reference transactions for feature engineering ───────────────────
# Transactions dated AFTER reference_date are FUTURE DATA relative to prediction.
# Using them would directly leak the target (active merchant = not churning).
# This is an undocumented trap in addition to T1-T5.
pre = df[df["transaction_date"] <= REF_DATE].copy()

def agg_window(data, window_start, suffix):
    w = data[data["transaction_date"] >= window_start]
    return w.groupby("merchant_id", observed=True).agg(**{
        f"tpv_{suffix}":           ("amount", "sum"),
        f"n_tx_{suffix}":          ("transaction_id", "count"),
        f"approval_rate_{suffix}": ("status", lambda x: (x == "approved").mean()),
        f"pct_ecom_{suffix}":      ("channel", lambda x: (x == "ecom").mean()),
        f"avg_tx_{suffix}":        ("amount", "mean"),
    })

all_time = pre.groupby("merchant_id", observed=True).agg(
    tpv_total           =("amount", "sum"),
    n_tx_total          =("transaction_id", "count"),
    approval_rate_total =("status", lambda x: (x == "approved").mean()),
    n_months_active     =("transaction_date", lambda x: x.dt.to_period("M").nunique()),
)

feats_3m = agg_window(pre, WIN_3M, "3m")
feats_6m = agg_window(pre, WIN_6M, "6m")

# ── Complaint feature (cap at reference_date to prevent T2 leakage) ──────────
complaints = (
    df.groupby("merchant_id", observed=True)["last_complaint_date"]
    .first()
    .reset_index()
)
complaints["lcd_capped"] = complaints["last_complaint_date"].where(
    complaints["last_complaint_date"] <= REF_DATE
)
complaints["days_since_complaint"] = (REF_DATE - complaints["lcd_capped"]).dt.days
complaints = complaints.set_index("merchant_id")[["days_since_complaint"]]

# ── Merchant metadata (snapshot at reference_date) ────────────────────────────
meta = df.drop_duplicates("merchant_id").set_index("merchant_id")[["segment", "mcc", "fla_churn90"]]

# ── Join everything ───────────────────────────────────────────────────────────
merchant_df = (
    all_time
    .join(feats_3m, how="left")
    .join(feats_6m, how="left")
    .join(complaints, how="left")
    .join(meta, how="left")
    .reset_index()
)

# ── Derived ratio features ────────────────────────────────────────────────────
merchant_df["tpv_trend_3m_6m"] = (
    merchant_df["tpv_3m"] / merchant_df["tpv_6m"].where(merchant_df["tpv_6m"] > 0)
).fillna(1.0).clip(0, 5)

merchant_df["log_tpv_total"] = np.log1p(merchant_df["tpv_total"])
merchant_df["log_tpv_3m"]    = np.log1p(merchant_df["tpv_3m"].fillna(0))

print(f"Feature matrix: {merchant_df.shape}")
print(f"Null counts:\n{merchant_df.isnull().sum()[merchant_df.isnull().sum() > 0]}")
print(f"\nClass balance:\n{merchant_df['fla_churn90'].value_counts()}")

# EXCLUDED features and reason:
# - cancellation_reason  → T1: filled only for churners → direct leakage
# - last_complaint_date  → T2: raw dates > ref_date → temporal leakage
# - transaction_date     → temporal identifier, not a feature
# - reference_date       → constant (same for all rows)
# - transaction_id       → surrogate key

Feature matrix: (9967, 22)
Null counts:
tpv_3m                  3005
n_tx_3m                 3005
approval_rate_3m        3005
pct_ecom_3m             3005
avg_tx_3m               3005
tpv_6m                  1142
n_tx_6m                 1142
approval_rate_6m        1142
pct_ecom_6m             1142
avg_tx_6m               1142
days_since_complaint    7156
dtype: int64

Class balance:
fla_churn90
0    9096
1     871
Name: count, dtype: Int64


## 3 · Split — temporal-aware

Un split aleatorio puede filtrar información del futuro. ¿Cómo lo evitas? Justifica.

In [4]:
# ── Why not a strict temporal split? ─────────────────────────────────────────
# All merchants share the SAME reference_date (2025-09-30) — it is a single snapshot.
# There is no second snapshot available to hold out, so a merchant-level stratified
# split is the correct approach. The temporal risk is controlled differently:
#   a) features are engineered ONLY from tx <= reference_date (above)
#   b) a rolling time-series CV would be ideal in production, but requires multiple
#      reference snapshots which this dataset does not provide.

TARGET = "fla_churn90"
FEATURES_NUM = [
    "tpv_total", "n_tx_total", "approval_rate_total", "n_months_active",
    "tpv_3m", "n_tx_3m", "approval_rate_3m", "pct_ecom_3m", "avg_tx_3m",
    "tpv_6m", "n_tx_6m", "approval_rate_6m", "pct_ecom_6m", "avg_tx_6m",
    "tpv_trend_3m_6m", "log_tpv_total", "log_tpv_3m", "days_since_complaint",
]
FEATURES_CAT = ["segment", "mcc"]

X = merchant_df[FEATURES_NUM + FEATURES_CAT].copy()
y = merchant_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {len(X_train):,} merchants  |  positives: {y_train.sum()} ({y_train.mean():.2%})")
print(f"Test:  {len(X_test):,}  merchants  |  positives: {y_test.sum()}  ({y_test.mean():.2%})")

Train: 7,973 merchants  |  positives: 697 (8.74%)
Test:  1,994  merchants  |  positives: 174  (8.73%)


## 4 · Pipeline + entrenamiento

In [5]:
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, FEATURES_NUM),
    ("cat", cat_pipe, FEATURES_CAT),
], remainder="drop")

# LightGBM with class imbalance handling via scale_pos_weight
# scale_pos_weight ≈ n_neg / n_pos ≈ 10.4 (inverse of churn rate)
pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())

clf = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

pipeline = Pipeline([("prep", preprocessor), ("clf", clf)])
pipeline.fit(X_train, y_train)

print(f"Model trained. scale_pos_weight = {pos_weight}")
print(f"Feature names: {FEATURES_NUM + FEATURES_CAT}")

Model trained. scale_pos_weight = 10
Feature names: ['tpv_total', 'n_tx_total', 'approval_rate_total', 'n_months_active', 'tpv_3m', 'n_tx_3m', 'approval_rate_3m', 'pct_ecom_3m', 'avg_tx_3m', 'tpv_6m', 'n_tx_6m', 'approval_rate_6m', 'pct_ecom_6m', 'avg_tx_6m', 'tpv_trend_3m_6m', 'log_tpv_total', 'log_tpv_3m', 'days_since_complaint', 'segment', 'mcc']


## 5 · Métricas

Para target con ~8% positivos, accuracy no sirve. Reporta:
- ROC-AUC
- PR-AUC (Average Precision)
- recall@k (k = 1%, 5%, 10% del test set)
- Calibración (Brier score o reliability diagram)

In [6]:
y_prob = pipeline.predict_proba(X_test)[:, 1]

roc_auc  = roc_auc_score(y_test, y_prob)
pr_auc   = average_precision_score(y_test, y_prob)
brier    = brier_score_loss(y_test, y_prob)

# Recall@k: what fraction of churners do we catch in the top k% ranked?
# LightGBM with num_leaves=31 over ~2k test merchants can easily produce tied
# scores (same leaf). A plain argsort breaks ties by array position, which is
# an artifact of X_test's row order, not risk — that made recall_at_1% noisy
# (a handful of tied merchants swapping in/out of the top-20 cutoff swings it
# by several points). np.lexsort with a fixed secondary key (original index)
# makes the ranking deterministic and reproducible across runs.
def recall_at_k(y_true, y_scores, k):
    n = max(1, int(len(y_true) * k))
    order = np.lexsort((np.arange(len(y_scores)), -np.asarray(y_scores)))
    top_idx = order[:n]
    return y_true.iloc[top_idx].sum() / y_true.sum()

r1  = recall_at_k(y_test, y_prob, 0.01)
r5  = recall_at_k(y_test, y_prob, 0.05)
r10 = recall_at_k(y_test, y_prob, 0.10)

metrics = {
    "roc_auc": round(roc_auc, 4),
    "pr_auc":  round(pr_auc, 4),
    "brier_score": round(brier, 4),
    "recall_at_1pct":  round(r1, 4),
    "recall_at_5pct":  round(r5, 4),
    "recall_at_10pct": round(r10, 4),
    "n_test": len(y_test),
    "n_positives_test": int(y_test.sum()),
}

print("=== METRICS ===")
for k, v in metrics.items():
    print(f"  {k:<22}: {v}")

(OUTPUTS / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("\nSaved to outputs/metrics.json")

if roc_auc > 0.95:
    print("\n⚠️  AUC > 0.95 — investigate for leakage before celebrating.")

=== METRICS ===
  roc_auc               : 0.5828
  pr_auc                : 0.1077
  brier_score           : 0.1248
  recall_at_1pct        : 0.0172
  recall_at_5pct        : 0.069
  recall_at_10pct       : 0.1379
  n_test                : 1994
  n_positives_test      : 174

Saved to outputs/metrics.json


## 6 · Interpretabilidad

Top-5 features importantes con justificación. SHAP valorable.

> Recuerda: importancia (SHAP, feature_importances) **NO es causalidad**. Documéntalo si toca.

In [7]:
all_feature_names = FEATURES_NUM + FEATURES_CAT

# SHAP on the LightGBM booster directly (after preprocessing)
X_test_prep = pipeline.named_steps["prep"].transform(X_test)
explainer = shap.TreeExplainer(pipeline.named_steps["clf"])
shap_values = explainer.shap_values(X_test_prep)

# For binary classification, shap_values may be a list [neg_class, pos_class]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Mean absolute SHAP — global importance
mean_abs_shap = np.abs(sv).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

print("=== TOP-10 FEATURES BY MEAN |SHAP| ===")
print(importance_df.head(10).to_string(index=False))

importance_df.to_csv(OUTPUTS / "feature_importance.csv", index=False)
print("\nSaved to outputs/feature_importance.csv")

top5 = importance_df.head(5)["feature"].tolist()
print(f"\nTop-5: {top5}")

# Interpretation generated FROM the actual top5 list, not hardcoded prose —
# a static narrative here previously listed features (segment,
# approval_rate_3m) that weren't actually in the computed top5, and omitted
# ones that were (tpv_trend_3m_6m, avg_tx_6m), because it was written before
# the final run and never re-synced.
FEATURE_STORY = {
    "days_since_complaint": "Recent complaint = strong churn signal.",
    "tpv_total": "Volume proxy: low all-time volume merchants churn more.",
    "tpv_trend_3m_6m": "Falling 3m-vs-6m TPV trend = early disengagement signal.",
    "avg_tx_6m": "Lower average ticket size over 6m = declining engagement.",
    "n_tx_total": "Activity tenure: fewer all-time transactions = higher risk.",
    "n_months_active": "Activity tenure: fewer active months = higher risk.",
    "tpv_6m": "Volume proxy: low 6m volume merchants churn more.",
    "approval_rate_total": "Declining approval = technical/fraud problems.",
    "approval_rate_3m": "Declining recent approval = technical/fraud problems.",
    "log_tpv_total": "Volume proxy (log-scaled): low-volume merchants churn more.",
    "avg_tx_3m": "Lower average ticket size over 3m = declining engagement.",
    "segment": "SMB structural risk vs Enterprise stickiness.",
    "mcc": "Merchant category correlates with churn risk.",
}

print("\nInterpretación:")
for feat in top5:
    story = FEATURE_STORY.get(feat, "(sin story documentada — añadir a FEATURE_STORY)")
    print(f"  {feat:<25} → {story}")

=== TOP-10 FEATURES BY MEAN |SHAP| ===
             feature  mean_abs_shap
days_since_complaint       0.936140
           tpv_total       0.233505
     tpv_trend_3m_6m       0.224539
           avg_tx_6m       0.214916
          n_tx_total       0.197017
              tpv_6m       0.180290
 approval_rate_total       0.164181
       log_tpv_total       0.154711
           avg_tx_3m       0.138544
                 mcc       0.133385

Saved to outputs/feature_importance.csv

Top-5: ['days_since_complaint', 'tpv_total', 'tpv_trend_3m_6m', 'avg_tx_6m', 'n_tx_total']

Interpretación:
  days_since_complaint      → Recent complaint = strong churn signal.
  tpv_total                 → Volume proxy: low all-time volume merchants churn more.
  tpv_trend_3m_6m           → Falling 3m-vs-6m TPV trend = early disengagement signal.
  avg_tx_6m                 → Lower average ticket size over 6m = declining engagement.
  n_tx_total                → Activity tenure: fewer all-time transactions = higher 

## 7 · Persistencia

Guarda el modelo entrenado en `outputs/model.pkl` (joblib) y un mini `model_card.md` con: features, target, métricas, supuestos, limitaciones.

In [8]:
joblib.dump(pipeline, OUTPUTS / "model.pkl")
print("Saved: outputs/model.pkl")

model_card = f"""# Model Card — Merchant Churn Classifier

## Model
- **Type**: LightGBM binary classifier
- **Target**: `fla_churn90` (churn within 90 days of reference date)
- **Reference date**: 2025-09-30

## Features ({len(all_feature_names)} total)
- **Numeric**: {', '.join(FEATURES_NUM)}
- **Categorical**: {', '.join(FEATURES_CAT)}

## Training Data
- {len(X_train):,} merchants in train, {len(X_test):,} in test (80/20 stratified)
- Churn rate: ~8.75% (1:10.4 imbalance handled via `scale_pos_weight={pos_weight}`)

## Metrics (test set)
- ROC-AUC:       {metrics['roc_auc']}
- PR-AUC:        {metrics['pr_auc']}
- Brier score:   {metrics['brier_score']}
- Recall@1%:     {metrics['recall_at_1pct']}
- Recall@5%:     {metrics['recall_at_5pct']}
- Recall@10%:    {metrics['recall_at_10pct']}

## Key Exclusions (leakage prevention)
- `cancellation_reason` — T1: filled only for churners (direct leakage)
- `last_complaint_date` raw — T2: some dates are post-reference (temporal leakage)
- Transactions dated > reference_date — undocumented trap: future activity

## Persistence
`outputs/model.pkl` is a `joblib`-pickled sklearn `Pipeline`. Pickle
deserialization executes arbitrary code — only `joblib.load()` this file if
you trust its provenance (e.g. you built it yourself from this repo). Not
meant for loading from an untrusted source.

## Limitations
1. **Discrimination is weak (ROC-AUC {metrics['roc_auc']} ≈ near-random)** — only
   safe to use for coarse deprioritization at high k (≥10%), not for precise
   targeting at low k. At recall@1% ({metrics['recall_at_1pct']}), the model
   catches essentially none of the churners in the top 1% ranked — not
   meaningfully better than a random selection of that size.
2. Single snapshot — no temporal CV possible with this dataset. Train/test is
   a stratified split of merchants sharing the same reference_date, not a
   held-out future period, so these metrics don't validate generalization to
   a genuinely future snapshot — only to unseen merchants from the same period.
3. Synthetic data — distribution may not match production Brazil merchants
4. TPV trend limited to 3m/6m windows; longer lookbacks could improve signal
5. No calibration step applied — probabilities may be miscalibrated
"""

(OUTPUTS / "model_card.md").write_text(model_card)
print("Saved: outputs/model_card.md")
print("\n=== MODEL CARD PREVIEW ===")
print(model_card)

Saved: outputs/model.pkl
Saved: outputs/model_card.md

=== MODEL CARD PREVIEW ===
# Model Card — Merchant Churn Classifier

## Model
- **Type**: LightGBM binary classifier
- **Target**: `fla_churn90` (churn within 90 days of reference date)
- **Reference date**: 2025-09-30

## Features (20 total)
- **Numeric**: tpv_total, n_tx_total, approval_rate_total, n_months_active, tpv_3m, n_tx_3m, approval_rate_3m, pct_ecom_3m, avg_tx_3m, tpv_6m, n_tx_6m, approval_rate_6m, pct_ecom_6m, avg_tx_6m, tpv_trend_3m_6m, log_tpv_total, log_tpv_3m, days_since_complaint
- **Categorical**: segment, mcc

## Training Data
- 7,973 merchants in train, 1,994 in test (80/20 stratified)
- Churn rate: ~8.75% (1:10.4 imbalance handled via `scale_pos_weight=10`)

## Metrics (test set)
- ROC-AUC:       0.5828
- PR-AUC:        0.1077
- Brier score:   0.1248
- Recall@1%:     0.0172
- Recall@5%:     0.069
- Recall@10%:    0.1379

## Key Exclusions (leakage prevention)
- `cancellation_reason` — T1: filled only for churne